# Dobot + Laser Manual Experiment Panel

Notebook cells are organized around the same logic as `experiment.py`: initialize once, use separate hardware on/off cells, then run robot steps with an endpoint detector.

## 1. Initialize Notebook State

In [ ]:
import time
from time import sleep

import config
from Dobot import (
    build_y_step_target,
    calculate_step_count,
    connect_robot,
    disconnect_robot,
    has_robot_error,
    move_linear_point,
    move_relative_xyz,
    prepare_robot,
    return_to_pose,
    send_do_pulse,
    start_feedback,
    turn_do_off,
)
from Laser import connect_laser
from experiment import initialize_laser_from_config

# Hardware handles.
dobot = None
laser = None
feed_thread = None

# Experiment state.
saved_start_pose = None
endpoint_pose = None
endpoint_reached = False
laser_running = False
current_step = 0
step_count = calculate_step_count(config.TOTAL_DISTANCE_MM, config.STEP_DISTANCE_MM)
do_records = []
endpoint_records = []


def require_robot():
    if dobot is None:
        raise RuntimeError("Run the Dobot ON cell first")


def require_laser():
    if laser is None:
        raise RuntimeError("Run the Laser ON cell first")


def save_original_pose():
    global saved_start_pose, endpoint_pose, endpoint_reached, current_step
    require_robot()
    saved_start_pose = dobot.GetCurrentPose()
    endpoint_pose = None
    endpoint_reached = False
    current_step = 0
    print("Original pose saved:", saved_start_pose)
    print("Step count:", step_count)


def detect_endpoint():
    global endpoint_pose, endpoint_reached
    if current_step < step_count:
        return False

    endpoint_reached = True
    endpoint_pose = dobot.GetCurrentPose()
    endpoint_records.append({
        "step": current_step,
        "endpoint_pose": endpoint_pose,
    })
    print("Endpoint detected at step", current_step)
    print("Endpoint pose:", endpoint_pose)
    return True


def stop_laser_if_running():
    global laser_running
    if laser is not None and laser_running:
        laser.stop()
        laser_running = False
        print("Laser stopped")


def return_to_original_location():
    if saved_start_pose is None:
        raise RuntimeError("No original pose saved")
    require_robot()
    return_to_pose(
        dobot,
        saved_start_pose,
        config.SPEED_RATIO,
        do_indexes=[config.TRIGGER_DO_INDEX],
    )


def handle_endpoint_if_needed():
    if detect_endpoint():
        stop_laser_if_running()
        return_to_original_location()
        return True
    return False


def print_config_summary():
    print("Dobot IP:", config.DOBOT_IP)
    print("Speed ratio:", config.SPEED_RATIO)
    print("Laser DLL path:", config.LASER_DLL_PATH)
    print("Laser wavelength:", config.LASER_WAVELENGTH_NM, "nm")
    print("Laser mode:", config.LASER_FIRE_MODE)
    print("Step distance:", config.STEP_DISTANCE_MM, "mm")
    print("Total distance:", config.TOTAL_DISTANCE_MM, "mm")
    print("Step wait:", config.STEP_WAIT_SECONDS, "s")


print_config_summary()
print("Notebook initialized")

## 2. Hardware On / Off Cells

### Dobot ON

In [ ]:
if dobot is None:
    dobot = connect_robot(config.DOBOT_IP)

if not prepare_robot(dobot, config.SPEED_RATIO):
    raise RuntimeError("Dobot prepare failed")

if has_robot_error(dobot):
    raise RuntimeError("Dobot has active errors")

if feed_thread is None:
    feed_thread = start_feedback(dobot)
    sleep(1)

save_original_pose()
print("Dobot ON and ready")

### Dobot OFF / Return Original

In [ ]:
if dobot is not None:
    stop_laser_if_running()
    if saved_start_pose is not None:
        return_to_original_location()
    turn_do_off(dobot, config.TRIGGER_DO_INDEX)
    disconnect_robot(dobot)
    dobot = None
    feed_thread = None
    saved_start_pose = None
    print("Dobot returned, trigger DO off, and connection closed")
else:
    print("Dobot is not connected")


### Laser ON / Initialize

In [ ]:
if laser is None:
    laser = connect_laser(config.LASER_DLL_PATH)

initialize_laser_from_config(laser)
print("Laser ON and initialized")

### Run Laser Manually


In [ ]:
require_laser()

# Action options:
#   run           -> start laser, leave it running
#   stop          -> stop laser
#   pulse <sec>   -> run laser for <sec> seconds then stop (default 1s)
action = input("Laser action [run / stop / pulse <seconds>]: ").strip().lower()

if action == "run":
    laser.run()
    laser_running = True
    print("Laser running. Re-run this cell with 'stop' to stop it.")
elif action == "stop":
    laser.stop()
    laser_running = False
    print("Laser stopped.")
elif action.startswith("pulse"):
    parts = action.split()
    duration_s = float(parts[1]) if len(parts) > 1 else 1.0
    laser.run()
    laser_running = True
    print(f"Laser running for {duration_s}s...")
    sleep(duration_s)
    laser.stop()
    laser_running = False
    print("Laser stopped after pulse.")
else:
    print("Unknown action. Use: run, stop, or 'pulse <seconds>'.")


### Set Laser Wavelength


In [ ]:
require_laser()

wavelength_text = input(
    f"Laser wavelength nm [{config.LASER_WAVELENGTH_NM}]: "
).strip()

if wavelength_text:
    new_wavelength_nm = float(wavelength_text)
else:
    new_wavelength_nm = config.LASER_WAVELENGTH_NM

laser.set_wavelength(new_wavelength_nm)
print("Laser wavelength set to", new_wavelength_nm, "nm")


### Laser OFF / Close

In [ ]:
if laser is not None:
    stop_laser_if_running()
    laser.close()
    laser = None
    print("Laser closed")
else:
    print("Laser is not connected")

### Manual Move By cm (dx, dy, dz)


In [ ]:
require_robot()

# Input: dx, dy, dz in cm (sign = direction).
# Examples:
#   3, 0, 3   -> +X 3cm, +Z 3cm
#   1, 0, -1  -> +X 1cm, -Z 1cm
#   -2, 0, 0  -> -X 2cm
text = input("Move dx, dy, dz in cm: ")
dx_cm, dy_cm, dz_cm = [float(v.strip()) for v in text.split(",")]

target_pose = move_relative_xyz(
    dobot,
    dx=dx_cm * 10,
    dy=dy_cm * 10,
    dz=dz_cm * 10,
    speed_ratio=config.SPEED_RATIO,
)
print(f"Moved by cm: ({dx_cm}, {dy_cm}, {dz_cm})")
print("Moved to:", target_pose)


### Manual Move By Direction (default 3cm step)


In [ ]:
require_robot()

# Input: dir_x, dir_y, dir_z. Each is -1, 0, or 1. Step length is 3 cm.
# Examples:
#   1, 0, -1  -> +X 3cm, -Z 3cm
#   0, 1, 0   -> +Y 3cm
#   -1, 0, 0  -> -X 3cm
STEP_CM = 3
text = input("Direction dir_x, dir_y, dir_z (each -1/0/1): ")
dir_x, dir_y, dir_z = [int(v.strip()) for v in text.split(",")]

for value in (dir_x, dir_y, dir_z):
    if value not in (-1, 0, 1):
        raise ValueError("Each direction must be -1, 0, or 1")

target_pose = move_relative_xyz(
    dobot,
    dx=dir_x * STEP_CM * 10,
    dy=dir_y * STEP_CM * 10,
    dz=dir_z * STEP_CM * 10,
    speed_ratio=config.SPEED_RATIO,
)
print(f"Moved direction ({dir_x}, {dir_y}, {dir_z}) by {STEP_CM}cm each")
print("Moved to:", target_pose)


## 3. Experiment Step Cells

### Laser Toggle With Endpoint Guard

In [ ]:
require_laser()

if endpoint_reached:
    stop_laser_if_running()
    print("Endpoint already reached; laser remains off")
elif laser_running:
    laser.stop()
    laser_running = False
    print("Laser manually stopped")
else:
    laser.run()
    laser_running = True
    print("Laser running until endpoint is detected")

### Run Robot Once

In [ ]:
require_robot()

if saved_start_pose is None:
    save_original_pose()

if endpoint_reached:
    print("Endpoint already reached. Returning to original location.")
    stop_laser_if_running()
    return_to_original_location()
elif current_step >= step_count:
    handle_endpoint_if_needed()
else:
    current_step += 1
    pulse_start = time.perf_counter()
    do_on_result, do_off_result = send_do_pulse(
        dobot,
        config.TRIGGER_DO_INDEX,
        config.TRIGGER_PULSE_SECONDS,
    )
    do_records.append({
        "step": current_step,
        "on_result": do_on_result,
        "off_result": do_off_result,
    })

    target_pose = build_y_step_target(saved_start_pose, current_step, config.STEP_DISTANCE_MM)
    print(f"Step {current_step}/{step_count}: moving to Y={target_pose[1]:.3f}")
    move_linear_point(dobot, target_pose, config.SPEED_RATIO)
    print("Pulse duration ms:", int((time.perf_counter() - pulse_start) * 1000))

    if not handle_endpoint_if_needed():
        sleep(config.STEP_WAIT_SECONDS)

print("Current step:", current_step)
print("Endpoint reached:", endpoint_reached)

### Endpoint Detector

In [ ]:
require_robot()

print("Current step:", current_step, "/", step_count)
print("Endpoint reached:", endpoint_reached)

if saved_start_pose is not None:
    current_pose = dobot.GetCurrentPose()
    expected_endpoint_y = saved_start_pose[1] - config.TOTAL_DISTANCE_MM
    print("Current pose:", current_pose)
    print("Expected endpoint Y:", expected_endpoint_y)

if handle_endpoint_if_needed():
    print("Endpoint handled: laser stopped and Dobot returned")
else:
    print("Endpoint not reached yet")

### Reset Run State

In [ ]:
require_robot()
stop_laser_if_running()
save_original_pose()
do_records = []
endpoint_records = []
print("Run state reset")

## 4. Records

In [ ]:
print("DO records:", len(do_records))
print(do_records)
print("Endpoint records:", endpoint_records)